# 🦺 Smart PPE Compliance Monitoring System
## Training Notebook — YOLO26s-seg on Kaggle

---

### 📋 Classes
| ID | Class |
|---|---|
| 0 | HelmetVestGloves |
| 1 | No-Gloves |
| 2 | No-Helmet |
| 3 | No-Shoes |
| 4 | No-Vest |
| 5 | Shoes |

---
> ⚡ **مهم:** تأكد إن Accelerator = **GPU P100**
>
> `Settings → Accelerator → GPU P100`

## 📦 Step 1 — Install Dependencies

In [ ]:
# Install required packages
!pip install ultralytics roboflow -q

# Verify GPU
import torch
print(f'✅ PyTorch version: {torch.__version__}')
print(f'✅ CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
    print(f'✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

## 📥 Step 2 — Download Dataset from Roboflow

In [ ]:
from roboflow import Roboflow
import os

# ⚠️ حط الـ API Key بتاعتك هنا
rf = Roboflow(api_key="YOUR_API_KEY")

# PPE Dataset — dataset-ppe-polygon
project = rf.workspace("ppe-instance-segmentation-jvlwi").project("dataset-ppe-polygon-zzpu8")
version = project.version(2)
dataset = version.download("yolov8-seg")  # YOLO26 compatible with yolov8-seg format

print(f'✅ Dataset downloaded to: {dataset.location}')

## 🔍 Step 3 — Explore Dataset

In [ ]:
import yaml
from pathlib import Path

# Load data.yaml
data_yaml_path = Path(dataset.location) / 'data.yaml'
with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

print('📊 Dataset Info:')
print(f'  Classes ({data_config["nc"]}): {data_config["names"]}')

# Count images
for split in ['train', 'valid', 'test']:
    split_path = Path(dataset.location) / split / 'images'
    if split_path.exists():
        count = len(list(split_path.glob('*.jpg')) + list(split_path.glob('*.png')))
        print(f'  {split}: {count} images')

In [ ]:
# Visualize sample images with annotations
import cv2
import matplotlib.pyplot as plt
import numpy as np
import random

CLASS_NAMES = ['HelmetVestGloves', 'No-Gloves', 'No-Helmet', 'No-Shoes', 'No-Vest', 'Shoes']

COLORS = [
    (0, 255, 0),    # HelmetVestGloves - green
    (128, 0, 128),  # No-Gloves - purple
    (0, 0, 255),    # No-Helmet - red
    (0, 165, 255),  # No-Shoes - orange
    (0, 0, 128),    # No-Vest - dark red
    (255, 255, 0),  # Shoes - cyan
]

train_img_path = Path(dataset.location) / 'train' / 'images'
train_lbl_path = Path(dataset.location) / 'train' / 'labels'

all_imgs = list(train_img_path.glob('*.jpg')) + list(train_img_path.glob('*.png'))
sample_imgs = random.sample(all_imgs, min(6, len(all_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('📊 Sample Training Images with Annotations', fontsize=16)

for idx, img_path in enumerate(sample_imgs):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    lbl_path = train_lbl_path / (img_path.stem + '.txt')
    if lbl_path.exists():
        with open(lbl_path) as f:
            for line in f:
                parts = list(map(float, line.strip().split()))
                cls = int(parts[0])
                # Polygon points (segmentation format)
                poly_pts = parts[1:]
                if len(poly_pts) >= 4:
                    pts = np.array(poly_pts).reshape(-1, 2)
                    pts[:, 0] *= w
                    pts[:, 1] *= h
                    pts = pts.astype(np.int32)
                    color = COLORS[cls % len(COLORS)]
                    cv2.polylines(img, [pts], True, color, 2)
                    x1, y1 = pts[0]
                    cv2.putText(img, CLASS_NAMES[cls], (x1, y1 - 5),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    ax = axes[idx // 3][idx % 3]
    ax.imshow(img)
    ax.axis('off')

plt.tight_layout()
plt.savefig('sample_annotations.png', dpi=100, bbox_inches='tight')
plt.show()
print('✅ Sample images saved!')

## 🚀 Step 4 — Train YOLO26s-seg

In [ ]:
from ultralytics import YOLO

# Load YOLO26s-seg pretrained model
model = YOLO('yolo26s-seg.pt')

print('🚀 Starting Training...')
print('Model: YOLO26s-seg')
print(f'Dataset: {data_yaml_path}')

# Train
results = model.train(
    data=str(data_yaml_path),
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,           # Early stopping
    lr0=0.01,              # Initial learning rate
    lrf=0.01,              # Final learning rate
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    warmup_momentum=0.8,
    hsv_h=0.015,           # Augmentation
    hsv_s=0.7,
    hsv_v=0.4,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    name='ppe_yolo26s_seg',
    project='/kaggle/working/PPE_Monitoring',
    exist_ok=True,
    device=0,              # GPU
    workers=2,             # Kaggle workers
    amp=True,              # Mixed precision (faster)
    verbose=True,
    save=True,
    save_period=10,        # Save checkpoint every 10 epochs
)

print('\n✅ Training Complete!')
print(f'Best weights saved at: {results.save_dir}/weights/best.pt')

## 📊 Step 5 — Evaluate Model

In [ ]:
from ultralytics import YOLO
from pathlib import Path

# Load best model
best_model_path = '/kaggle/working/PPE_Monitoring/ppe_yolo26s_seg/weights/best.pt'
model = YOLO(best_model_path)

# Validate on test set
metrics = model.val(
    data=str(data_yaml_path),
    split='test',
    imgsz=640,
    conf=0.25,
    iou=0.6,
    verbose=True
)

print('\n📊 Evaluation Results:')
print(f'  mAP50 (Box):      {metrics.box.map50:.4f}')
print(f'  mAP50-95 (Box):   {metrics.box.map:.4f}')
print(f'  mAP50 (Seg):      {metrics.seg.map50:.4f}')
print(f'  mAP50-95 (Seg):   {metrics.seg.map:.4f}')
print(f'  Precision:        {metrics.box.mp:.4f}')
print(f'  Recall:           {metrics.box.mr:.4f}')

In [ ]:
# Plot training curves
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

results_dir = Path('/kaggle/working/PPE_Monitoring/ppe_yolo26s_seg')

plots = [
    ('results.png', 'Training Results'),
    ('confusion_matrix_normalized.png', 'Confusion Matrix'),
    ('PR_curve.png', 'PR Curve'),
]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('📊 Model Evaluation', fontsize=16)

for ax, (filename, title) in zip(axes, plots):
    img_path = results_dir / filename
    if img_path.exists():
        img = mpimg.imread(str(img_path))
        ax.imshow(img)
        ax.set_title(title, fontsize=12)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 🧪 Step 6 — Test on Sample Images

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import random

# Violation classes (missing PPE)
VIOLATION_CLASSES = ['No-Gloves', 'No-Helmet', 'No-Shoes', 'No-Vest']

# Test on random images
test_img_path = Path(dataset.location) / 'test' / 'images'
all_test_imgs = list(test_img_path.glob('*.jpg')) + list(test_img_path.glob('*.png'))
sample_tests = random.sample(all_test_imgs, min(4, len(all_test_imgs)))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('🧪 PPE Segmentation Results', fontsize=16)

for idx, img_path in enumerate(sample_tests):
    preds = model.predict(
        str(img_path),
        conf=0.3,
        iou=0.5,
        verbose=False
    )

    result_img = preds[0].plot()
    result_img = cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB)

    # Count violations
    violations = []
    for box in preds[0].boxes:
        cls_name = model.names[int(box.cls)]
        if cls_name in VIOLATION_CLASSES:
            violations.append(cls_name)

    ax = axes[idx // 2][idx % 2]
    ax.imshow(result_img)
    status = f'⚠️ Violations: {violations}' if violations else '✅ Compliant'
    ax.set_title(status, fontsize=10, color='red' if violations else 'green')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 💾 Step 7 — Save Weights

In [ ]:
import shutil
from pathlib import Path

best_pt = Path('/kaggle/working/PPE_Monitoring/ppe_yolo26s_seg/weights/best.pt')
last_pt = Path('/kaggle/working/PPE_Monitoring/ppe_yolo26s_seg/weights/last.pt')

# Copy to /kaggle/working for easy access
shutil.copy(best_pt, '/kaggle/working/ppe_best.pt')
shutil.copy(last_pt, '/kaggle/working/ppe_last.pt')

print(f'✅ best.pt size: {best_pt.stat().st_size / 1024**2:.1f} MB')
print('✅ Weights copied to /kaggle/working/')
print('📥 يقدر تنزلهم من Output tab على اليمين')

## 📤 Step 8 — Export Model (Optional)

In [ ]:
# Export to ONNX for faster CPU inference
model = YOLO('/kaggle/working/PPE_Monitoring/ppe_yolo26s_seg/weights/best.pt')

# ONNX Export
model.export(format='onnx', imgsz=640, dynamic=True, simplify=True)
print('✅ ONNX model exported!')

# TFLite Export (for mobile/edge)
# model.export(format='tflite', imgsz=640)
# print('✅ TFLite model exported!')

---
## ✅ Training Complete!

### الخطوة الجاية:
1. روح **Output tab** على اليمين
2. حمّل `ppe_best.pt`
3. حطه في `models/` folder في المشروع

```
ppe-monitoring/
├── models/
│   └── ppe_best.pt  ← هنا
├── app.py
├── detector.py
└── database.py
```